In [ ]:
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# QUESTION 3 : INVERSION DES PRIX HW EN VOLATILITÉS IMPLICITES BLACK STANDARD
# -------------------------------------------------------------------------

# 1. Fonction de valorisation Black Standard (celle du marché, Partie 2)
def black_caplet_standard(sigma_blk, L, K, delta, B_Ti, Ti_fix, N=1):
    sqrtT = np.sqrt(Ti_fix)
    d1 = (np.log(L / K) + 0.5 * sigma_blk**2 * Ti_fix) / (sigma_blk * sqrtT)
    d2 = d1 - sigma_blk * sqrtT
    return N * delta * B_Ti * (L * norm.cdf(d1) - K * norm.cdf(d2))

# 2. Inversion de la fonction Black Standard par dichotomie
def implied_vol_standard(price_cible, L, K, delta, B_Ti, Ti_fix, N=1, tol=1e-6):
    s_low, s_high = 1e-4, 2.0
    for _ in range(100):
        s_mid = (s_low + s_high) / 2
        f_mid = black_caplet_standard(s_mid, L, K, delta, B_Ti, Ti_fix, N)
        if f_mid > price_cible:
            s_high = s_mid
        else:
            s_low = s_mid
        if s_high - s_low < tol:
            break
    return s_mid

# 3. Données de marché (Volatilités implicites issues du Tableau 1 / Section 2)
market_vols = [0.312, 0.284, 0.266, 0.250, 0.244, 0.250, 0.272]

# 4. Calcul des volatilités implicites du modèle Hull-White
hw_implied_vols = []

print("\n--- COMPARAISON DES SMILES DE VOLATILITÉ ---")
print(f"{'Strike (bps)':<15} {'Prix HW':<15} {'Vol. Impl. HW':<15} {'Vol. Impl. Marché':<15}")
print("-" * 65)

for s_bps, vol_mkt in zip(strikes_bps, market_vols):
    K_s = L_fwd + s_bps / 10000
    # On récupère le prix HW (déjà calculé à la Q2)
    price_HW = black_caplet_HW(sigma_i, L_fwd, K_s, delta, B_Ti, Ti, t, N)

    # On inverse ce prix dans le modèle de Black Standard
    vol_HW = implied_vol_standard(price_HW, L_fwd, K_s, delta, B_Ti, Ti_fix, N)
    hw_implied_vols.append(vol_HW)

    print(f"{s_bps:<15} {price_HW:.4%}       {vol_HW*100:.2f}%           {vol_mkt*100:.2f}%")

# 5. Tracé du graphique (Smile de volatilité)
plt.figure(figsize=(8, 5))
plt.plot(strikes_bps, [v * 100 for v in market_vols], marker='o', label='Marché (Smile)')
plt.plot(strikes_bps, [v * 100 for v in hw_implied_vols], marker='s', linestyle='--', label='Modèle Hull-White')

plt.title("Comparaison des Smiles de Volatilité : Marché vs Hull-White")
plt.xlabel("Strike Relatif (bps)")
plt.ylabel("Volatilité Implicite Black (%)")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='ATM')
plt.show()